# ETL: Entregable 1

In [1]:
from dotenv import dotenv_values
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta
import pandas as pd
from spotipy import Spotify
from spotipy.oauth2 import SpotifyOAuth
from helpers import check_if_valid_data, psql_insert_copy

In [2]:
config = dotenv_values(".env")

In [3]:
# Spotify settings
CLIENT_ID = config["CLIENT_ID"]
CLIENT_SECRET = config["CLIENT_SECRET"]
SPOTIPY_REDIRECT_URI = config["SPOTIPY_REDIRECT_URI"]
SCOPE = config["SCOPE"]

In [4]:
# DB settings
TABLENAME = config["TABLENAME"]
DB_USERNAME = config["DB_USERNAME"]
DB_PASSWORD = config["DB_PASSWORD"]
DB_HOST = config["DB_HOST"]
DB_PORT = config["DB_PORT"]
DB_NAME = config["DB_NAME"]
DB_SCHEMA = config["DB_SCHEMA"]

In [5]:
# Conexión a Spotify
sp = Spotify(auth_manager=SpotifyOAuth(client_id=CLIENT_ID,
                                               client_secret=CLIENT_SECRET,
                                               redirect_uri=SPOTIPY_REDIRECT_URI,
                                               scope=SCOPE))

# Conviertir una timestamp de unix en milisegundos
today = datetime.now()
yesterday = today - timedelta(days=1)
yesterday = yesterday.replace(hour=0, minute=0, second=0, microsecond=0)
yesterday_unix_timestamp = int(yesterday.timestamp()) * 1000
limit = 50    

# Retorna un diccionario con las canciones escuchadas
raw_data = sp.current_user_recently_played(limit=limit, after=yesterday_unix_timestamp)

In [6]:
raw_data.keys()

dict_keys(['items', 'next', 'cursors', 'limit', 'href'])

In [7]:
song_names = []
artist_names = []
played_at_list = []
timestamps = []

In [8]:
# Guardamos la data en listas
for song in raw_data["items"]:
    song_names.append(song["track"]["name"])
    artist_names.append(song["track"]["artists"][0]["name"])
    played_at_list.append(song["played_at"]),
    timestamps.append(song["played_at"][0:10])

In [9]:
# Creamos un diccionario de listas
song_dict = {
    "song_name" : song_names,
    "artist_name": artist_names,
    "played_at" : played_at_list,
    "timestamp" : timestamps
}

In [10]:
# Crear el DataFrame
song_df = pd.DataFrame(song_dict, columns = ["song_name",
                                             "artist_name",
                                             "played_at",
                                             "timestamp"])

In [11]:
song_df.head()

,song_name,artist_name,played_at,timestamp
0,Imposible,Callejeros,2023-06-10T00:01:55.648Z,2023-06-10
1,Solo voy,La 25,2023-06-09T23:42:58.214Z,2023-06-09
2,Funeral,Jóvenes Pordioseros,2023-06-09T23:36:53.544Z,2023-06-09
3,Esto no se ve,Jóvenes Pordioseros,2023-06-09T23:34:32.443Z,2023-06-09
4,Cuando Me Muera,Jóvenes Pordioseros,2023-06-09T23:30:57.119Z,2023-06-09


In [12]:
# Eliminar fechas diferentes a las que queremos
clean_song_df = song_df[pd.to_datetime(song_df["played_at"]).dt.date == yesterday.date()]

In [13]:
clean_song_df

,song_name,artist_name,played_at,timestamp
1,Solo voy,La 25,2023-06-09T23:42:58.214Z,2023-06-09
2,Funeral,Jóvenes Pordioseros,2023-06-09T23:36:53.544Z,2023-06-09
3,Esto no se ve,Jóvenes Pordioseros,2023-06-09T23:34:32.443Z,2023-06-09
4,Cuando Me Muera,Jóvenes Pordioseros,2023-06-09T23:30:57.119Z,2023-06-09
5,Inmovil,Cabezones,2023-06-09T23:21:32.728Z,2023-06-09
6,Muñeca Roja,Massacre,2023-06-09T23:15:40.895Z,2023-06-09
7,Diferentes Maneras - En Vivo,Massacre,2023-06-09T23:11:05.047Z,2023-06-09
8,Divorcio,Massacre,2023-06-09T23:06:50.307Z,2023-06-09
9,Ana No Duerme,Massacre,2023-06-09T23:02:29.733Z,2023-06-09
10,La Cita,Massacre,2023-06-09T22:59:18.441Z,2023-06-09


In [14]:
# Data validation
if check_if_valid_data(clean_song_df):
    print("Data valid, proceed to Load stage...")

Data valid, proceed to Load stage...


# Conexión a PostgreSQL en local

In [15]:
# Load

# Crear objeto engine para conecatarse a la base de datos
conn_string = f"postgresql+psycopg2://{DB_USERNAME}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_string)

In [16]:
# Utilizar el bloque with sirve para no preocuparnos de cerrar la conn a la db
# el bloque lo hace solo.
with engine.connect() as conn:
    # Ejecuta la consulta
    sql_query = text("""
    CREATE TABLE IF NOT EXISTS coder.my_tracks_history(
        id_my_tracks_history SERIAL NOT NULL,
        song_name VARCHAR(200) NOT NULL,
        artist_name VARCHAR(200) NOT NULL,
        played_at TIMESTAMP NOT NULL,
        timestamp DATE NOT NULL,

        CONSTRAINT pk_my_tracks_history PRIMARY KEY (played_at)
    );
    """)
    conn.execute(sql_query)

C:\Users\lautaro.poletto\AppData\Local\Temp\ipykernel_14192\3315831519.py:16: RemovedIn20Warning: Deprecated API features detected! These feature(s) are not compatible with SQLAlchemy 2.0. To prevent incompatible upgrades prior to updating applications, ensure requirements files are pinned to "sqlalchemy<2.0". Set environment variable SQLALCHEMY_WARN_20=1 to show all deprecation warnings.  Set environment variable SQLALCHEMY_SILENCE_UBER_WARNING=1 to silence this message. (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  conn.execute(sql_query)


In [18]:
rows_imported = 0
print(f"Importing rows {rows_imported} to {rows_imported + clean_song_df.shape[0]}... for table {TABLENAME}")

# Cargamos los nuevos datos a la ta
clean_song_df.to_sql(
    name="my_tracks_history",
    con=engine,
    schema="coder",
    if_exists="append",
    index=False,
    chunksize=10000,
    method=psql_insert_copy
)

rows_imported += clean_song_df.shape[0]
print(f"Importing rows {rows_imported} to {clean_song_df.shape[0]}... for table {TABLENAME}")
print(f"Data imported successful")

# close conn
engine.dispose()

Importing rows 0 to 30... for table my_tracks_history
Importing rows 30 to 30... for table my_tracks_history
Data imported successful


# Conexión a AWS Redshift

In [40]:
# # Load

# # AWS Redshift settings
# REDSHIFT_HOST = config["AWS_HOST"]
# REDSHIFT_PORT = int(config["AWS_PORT"])
# REDSHIFT_DATABASE = config["AWS_DB_NAME"]
# REDSHIFT_SCHEMA = config["AWS_DB_SCHEMA"]
# REDSHIFT_USERNAME = config["AWS_USER"]
# REDSHIFT_PASS = config["AWS_PASSWORD"]

In [41]:
engine = create_engine(f"redshift+psycopg2://{REDSHIFT_USERNAME}:{REDSHIFT_PASS}@{REDSHIFT_HOST}:{REDSHIFT_PORT}/{REDSHIFT_DATABASE}")
conn = engine.connect()

In [42]:
sql_query = """
CREATE TABLE IF NOT EXISTS lautaropoletto_coderhouse.my_tracks_history(
    id_my_tracks_history bigint identity(0, 1) NOT NULL,
    song_name VARCHAR(200) NOT NULL,
    artist_name VARCHAR(200) NOT NULL,
    played_at TIMESTAMP NOT NULL,
    timestamp DATE NOT NULL,
    primary key(played_at))
    distkey(id_my_tracks_history)
    compound sortkey(timestamp, artist_name);
"""

conn.execute(sql_query)

C:\Users\lautaro.poletto\AppData\Local\Temp\ipykernel_1488\2042616853.py:13: RemovedIn20Warning: Deprecated API features detected! These feature(s) are not compatible with SQLAlchemy 2.0. To prevent incompatible upgrades prior to updating applications, ensure requirements files are pinned to "sqlalchemy<2.0". Set environment variable SQLALCHEMY_WARN_20=1 to show all deprecation warnings.  Set environment variable SQLALCHEMY_SILENCE_UBER_WARNING=1 to silence this message. (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  conn.execute(sql_query)


In [44]:
rows_imported = 0
table_name = "my_tracks_history"

print(f"Importing rows {rows_imported} to {rows_imported + clean_song_df.shape[0]}... for table {table_name}")

clean_song_df.to_sql(table_name,
               con=engine,
               schema=REDSHIFT_SCHEMA,
               index=False,
               if_exists="append")

rows_imported += clean_song_df.shape[0]
print(f"Importing rows {rows_imported} to {clean_song_df.shape[0]}... for table {table_name}")
print(f"Data imported successful")

conn.close()

Importing rows 0 to 22... for table my_tracks_history
Importing rows 22 to 22... for table my_tracks_history
Data imported successful
